In [1]:
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIG
# ============================================================

DATA_DIR = "../../data_results/2_results/2-3_gppmax_cup_6methods"
FILE_TMPL = "calculated_GPPmax_CUP_ampFracs_10_20_30_25_thresholdScope_{sps}.xlsx"

OUT_DIR = "../../data_results/2_results/2-5_1_contribution_25"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_DETAIL_XLSX = os.path.join(OUT_DIR, "all_results_LMDI_detail.xlsx")

SPS_ECO = "ecosystem"
PFT_LIST = ["tree", "shrub", "sphagnum"]
SPS_LIST = [SPS_ECO] + PFT_LIST

PRETREAT_YEARS = (2011, 2013)
ANALYSIS_YEARS = (2017, 2021)

BASE_KEYS = ["co2", "warming", "year"]

COL_GPPMAX = "gpp_max_whsM"
COL_CUP = "cup_whsM_25"
COL_ALPHA = "alpha_whsM_25"

TEMP_COLS = ["temp_mean_anom", "temp_gs_mean_anom", "temp_max_anom"]


# ============================================================
# 2. LMDI FUNCTIONS
# ============================================================

def log_mean(x, y, eps=1e-12):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = np.maximum(x, eps)
    y = np.maximum(y, eps)

    out = np.empty_like(x)
    close = np.isclose(x, y, rtol=1e-12, atol=1e-15)

    out[close] = x[close]

    m = ~close
    out[m] = (x[m] - y[m]) / (np.log(x[m]) - np.log(y[m]))

    return out


def safe_ln_ratio(a1, a0, eps=1e-12):
    a1 = np.maximum(np.asarray(a1, dtype=float), eps)
    a0 = np.maximum(np.asarray(a0, dtype=float), eps)

    return np.log(a1 / a0)


def lmdi_additive_MCA(M0, C0, A0, M1, C1, A1, eps=1e-12):
    """
    GPP = GPPmax * CUP * alpha

    ΔGPP = ΔGPP_M + ΔGPP_C + ΔGPP_A
    """

    M0 = np.asarray(M0, dtype=float)
    C0 = np.asarray(C0, dtype=float)
    A0 = np.asarray(A0, dtype=float)

    M1 = np.asarray(M1, dtype=float)
    C1 = np.asarray(C1, dtype=float)
    A1 = np.asarray(A1, dtype=float)

    G0 = np.maximum(M0, eps) * np.maximum(C0, eps) * np.maximum(A0, eps)
    G1 = np.maximum(M1, eps) * np.maximum(C1, eps) * np.maximum(A1, eps)

    w = log_mean(G1, G0, eps=eps)

    dGPP_M = w * safe_ln_ratio(M1, M0, eps=eps)
    dGPP_C = w * safe_ln_ratio(C1, C0, eps=eps)
    dGPP_A = w * safe_ln_ratio(A1, A0, eps=eps)

    dGPP_total = G1 - G0
    resid = dGPP_total - (dGPP_M + dGPP_C + dGPP_A)

    return dGPP_total, dGPP_M, dGPP_C, dGPP_A, resid


# ============================================================
# 3. READ AND MERGE DATA
# ============================================================

def read_one_sps(sps):
    fp = os.path.join(DATA_DIR, FILE_TMPL.format(sps=sps))

    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing file: {fp}")

    df = pd.read_excel(fp)

    required = ["co2", "warming", "year", COL_GPPMAX, COL_CUP, COL_ALPHA]

    if sps == SPS_ECO:
        required += TEMP_COLS

    miss = [c for c in required if c not in df.columns]
    if miss:
        raise ValueError(f"[{sps}] missing columns: {miss}")

    keys = BASE_KEYS.copy()

    if "plot" in df.columns:
        keys = ["plot"] + keys

    keep_cols = keys.copy()

    if sps == SPS_ECO:
        keep_cols += TEMP_COLS

    keep_cols += [COL_GPPMAX, COL_CUP, COL_ALPHA]

    df = df[keep_cols].copy()

    df = df.rename(columns={
        COL_GPPMAX: f"{sps}_GPPmax",
        COL_CUP: f"{sps}_CUP",
        COL_ALPHA: f"{sps}_alpha",
    })

    return df


def merge_all_sps():
    dfs = [read_one_sps(sps) for sps in SPS_LIST]

    keys = BASE_KEYS.copy()

    if "plot" in dfs[0].columns:
        keys = ["plot"] + keys

    df = dfs[0].copy()

    for dfi in dfs[1:]:
        df = df.merge(dfi, on=keys, how="inner")

    return df, keys


# ============================================================
# 4. BASELINE
# ============================================================

def compute_baseline(df, keys_no_year):
    df0 = df[df["year"].between(PRETREAT_YEARS[0], PRETREAT_YEARS[1])].copy()

    base_cols = []

    for sps in SPS_LIST:
        base_cols += [
            f"{sps}_GPPmax",
            f"{sps}_CUP",
            f"{sps}_alpha",
        ]

    base = df0.groupby(keys_no_year, as_index=False)[base_cols].mean()

    rename_dict = {}

    for sps in SPS_LIST:
        rename_dict[f"{sps}_GPPmax"] = f"{sps}_GPPmax0"
        rename_dict[f"{sps}_CUP"] = f"{sps}_CUP0"
        rename_dict[f"{sps}_alpha"] = f"{sps}_alpha0"

    base = base.rename(columns=rename_dict)

    return base


# ============================================================
# 5. ADD LMDI TERMS
# ============================================================

def add_lmdi_terms(df, sps):
    M0 = df[f"{sps}_GPPmax0"]
    C0 = df[f"{sps}_CUP0"]
    A0 = df[f"{sps}_alpha0"]

    M1 = df[f"{sps}_GPPmax"]
    C1 = df[f"{sps}_CUP"]
    A1 = df[f"{sps}_alpha"]

    dGPP, dGPP_M, dGPP_C, dGPP_A, resid = lmdi_additive_MCA(
        M0, C0, A0,
        M1, C1, A1,
    )

    df[f"{sps}_dGPP"] = dGPP
    df[f"{sps}_dGPP_M"] = dGPP_M
    df[f"{sps}_dGPP_C"] = dGPP_C
    df[f"{sps}_dGPP_A"] = dGPP_A
    df[f"{sps}_resid"] = resid

    return df


# ============================================================
# 6. MAIN
# ============================================================

def main():

    # --------------------------------------------------------
    # 1. Read and merge ecosystem + PFT data
    # --------------------------------------------------------
    df, keys = merge_all_sps()

    keys_no_year = [k for k in keys if k != "year"]

    # --------------------------------------------------------
    # 2. Compute pretreatment baseline
    # --------------------------------------------------------
    baseline = compute_baseline(df, keys_no_year)

    df = df.merge(
        baseline,
        on=keys_no_year,
        how="left",
    )

    # --------------------------------------------------------
    # 3. LMDI contribution for ecosystem and PFTs
    # --------------------------------------------------------
    for sps in SPS_LIST:
        df = add_lmdi_terms(df, sps)

    # --------------------------------------------------------
    # 4. Keep analysis years only
    # --------------------------------------------------------
    df_analysis = df[
        df["year"].between(ANALYSIS_YEARS[0], ANALYSIS_YEARS[1])
    ].copy()

    # --------------------------------------------------------
    # 5. Save result
    # --------------------------------------------------------
    df_analysis.to_excel(OUT_DETAIL_XLSX, index=False)

    print("Saved LMDI contribution file:")
    print(OUT_DETAIL_XLSX)

    print("\nColumns related to LMDI contribution:")

    lmdi_cols = []

    for sps in SPS_LIST:
        lmdi_cols += [
            f"{sps}_dGPP",
            f"{sps}_dGPP_M",
            f"{sps}_dGPP_C",
            f"{sps}_dGPP_A",
            f"{sps}_resid",
        ]

    print(lmdi_cols)


if __name__ == "__main__":
    main()

Saved LMDI contribution file:
../../data_results/2_results/2-5_1_contribution_25/all_results_LMDI_detail.xlsx

Columns related to LMDI contribution:
['ecosystem_dGPP', 'ecosystem_dGPP_M', 'ecosystem_dGPP_C', 'ecosystem_dGPP_A', 'ecosystem_resid', 'tree_dGPP', 'tree_dGPP_M', 'tree_dGPP_C', 'tree_dGPP_A', 'tree_resid', 'shrub_dGPP', 'shrub_dGPP_M', 'shrub_dGPP_C', 'shrub_dGPP_A', 'shrub_resid', 'sphagnum_dGPP', 'sphagnum_dGPP_M', 'sphagnum_dGPP_C', 'sphagnum_dGPP_A', 'sphagnum_resid']


In [2]:
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIG
# ============================================================

IN_FILE = "../../data_results/2_results/2-5_1_contribution_25/all_results_LMDI_detail.xlsx"

OUT_DIR = "../../data_results/2_results/2-5_1_contribution_25/relative_contribution"
os.makedirs(OUT_DIR, exist_ok=True)

SPS_ECO = "ecosystem"
PFT_LIST = ["tree", "shrub", "sphagnum"]

GROUP_COLS = ["co2"]  # 可改为 ["co2", "warming"]


# ============================================================
# 2. BASIC FUNCTIONS
# ============================================================

def safe_divide(num, den):
    return np.where(den != 0, num / den, np.nan)


def weighted_signed_group(sub, comp_cols):
    """
    ξ_x = Σ_i(ΔGPP_x,i * |ΔGPP_i| / ΔGPP_i) / Σ_i|ΔGPP_i| * 100

    ΔGPP_i = sum of components for each row.
    """

    comps = sub[comp_cols].astype(float)
    dGPP_i = comps.sum(axis=1)

    valid = dGPP_i != 0
    denom = np.abs(dGPP_i[valid]).sum()

    out = {}

    if denom == 0:
        for col in comp_cols:
            out[col] = np.nan
        return out

    for col in comp_cols:
        numerator = (
            comps.loc[valid, col]
            * np.abs(dGPP_i[valid])
            / dGPP_i[valid]
        ).sum()

        out[col] = numerator / denom * 100.0

    return out


# ============================================================
# 3. METHOD 1: ABSOLUTE RELATIVE CONTRIBUTION
# ============================================================

def add_absolute_relative_detail(df):
    """
    RC_x = |x| / sum(|components|) * 100
    """

    # --------------------------------------------------------
    # 1) Ecosystem MCA
    # --------------------------------------------------------
    eco_cols = [
        "ecosystem_dGPP_M",
        "ecosystem_dGPP_C",
        "ecosystem_dGPP_A",
    ]

    denom = df[eco_cols].abs().sum(axis=1)

    df["ecosystem_abs_GPPmax_%"] = safe_divide(
        df["ecosystem_dGPP_M"].abs(), denom
    ) * 100.0

    df["ecosystem_abs_CUP_%"] = safe_divide(
        df["ecosystem_dGPP_C"].abs(), denom
    ) * 100.0

    df["ecosystem_abs_alpha_%"] = safe_divide(
        df["ecosystem_dGPP_A"].abs(), denom
    ) * 100.0

    # --------------------------------------------------------
    # 2) PFT GPP contribution to total PFT GPP change
    # --------------------------------------------------------
    pft_gpp_cols = [f"{pft}_dGPP" for pft in PFT_LIST]
    denom = df[pft_gpp_cols].abs().sum(axis=1)

    for pft in PFT_LIST:
        df[f"{pft}_abs_to_PFTsum_GPP_%"] = safe_divide(
            df[f"{pft}_dGPP"].abs(), denom
        ) * 100.0

    df["PFT_sum_dGPP"] = df[pft_gpp_cols].sum(axis=1)
    df["PFT_sum_minus_ecosystem_dGPP"] = (
        df["PFT_sum_dGPP"] - df["ecosystem_dGPP"]
    )

    # --------------------------------------------------------
    # 3) PFT internal MCA
    # --------------------------------------------------------
    for pft in PFT_LIST:
        cols = [
            f"{pft}_dGPP_M",
            f"{pft}_dGPP_C",
            f"{pft}_dGPP_A",
        ]

        denom = df[cols].abs().sum(axis=1)

        df[f"{pft}_abs_GPPmax_%"] = safe_divide(
            df[f"{pft}_dGPP_M"].abs(), denom
        ) * 100.0

        df[f"{pft}_abs_CUP_%"] = safe_divide(
            df[f"{pft}_dGPP_C"].abs(), denom
        ) * 100.0

        df[f"{pft}_abs_alpha_%"] = safe_divide(
            df[f"{pft}_dGPP_A"].abs(), denom
        ) * 100.0

    return df


def summarize_absolute_mean_std(df, group_cols=GROUP_COLS):
    rows_eco = []
    rows_pft_gpp = []
    rows_pft_mca = []

    for keys, sub in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)

        base = dict(zip(group_cols, keys))

        # ----------------------------------------------------
        # 1) Ecosystem MCA
        # ----------------------------------------------------
        row = base.copy()
        for var in ["GPPmax", "CUP", "alpha"]:
            col = f"ecosystem_abs_{var}_%"
            row[f"{var}_mean_%"] = sub[col].mean()
            row[f"{var}_std_%"] = sub[col].std()
        rows_eco.append(row)

        # ----------------------------------------------------
        # 2) PFT GPP contribution
        # ----------------------------------------------------
        row = base.copy()
        for pft in PFT_LIST:
            col = f"{pft}_abs_to_PFTsum_GPP_%"
            row[f"{pft}_mean_%"] = sub[col].mean()
            row[f"{pft}_std_%"] = sub[col].std()
        rows_pft_gpp.append(row)

        # ----------------------------------------------------
        # 3) PFT internal MCA
        # ----------------------------------------------------
        for pft in PFT_LIST:
            row = base.copy()
            row["PFT"] = pft

            for var in ["GPPmax", "CUP", "alpha"]:
                col = f"{pft}_abs_{var}_%"
                row[f"{var}_mean_%"] = sub[col].mean()
                row[f"{var}_std_%"] = sub[col].std()

            rows_pft_mca.append(row)

    return (
        pd.DataFrame(rows_eco),
        pd.DataFrame(rows_pft_gpp),
        pd.DataFrame(rows_pft_mca),
    )


# ============================================================
# 4. METHOD 2: SIGNED RELATIVE CONTRIBUTION
# ============================================================

def add_signed_relative_detail(df):
    """
    Row-level signed relative contribution.

    RC_x_signed = x / sum(components) * 100

    It keeps sign and can be >100% or <0%.
    """

    # --------------------------------------------------------
    # 1) Ecosystem MCA
    # --------------------------------------------------------
    eco_cols = [
        "ecosystem_dGPP_M",
        "ecosystem_dGPP_C",
        "ecosystem_dGPP_A",
    ]

    denom = df[eco_cols].sum(axis=1)

    df["ecosystem_signed_GPPmax_%"] = safe_divide(
        df["ecosystem_dGPP_M"], denom
    ) * 100.0

    df["ecosystem_signed_CUP_%"] = safe_divide(
        df["ecosystem_dGPP_C"], denom
    ) * 100.0

    df["ecosystem_signed_alpha_%"] = safe_divide(
        df["ecosystem_dGPP_A"], denom
    ) * 100.0

    # --------------------------------------------------------
    # 2) PFT GPP contribution to total PFT GPP change
    # --------------------------------------------------------
    pft_gpp_cols = [f"{pft}_dGPP" for pft in PFT_LIST]
    denom = df[pft_gpp_cols].sum(axis=1)

    for pft in PFT_LIST:
        df[f"{pft}_signed_to_PFTsum_GPP_%"] = safe_divide(
            df[f"{pft}_dGPP"], denom
        ) * 100.0

    # --------------------------------------------------------
    # 3) PFT internal MCA
    # --------------------------------------------------------
    for pft in PFT_LIST:
        cols = [
            f"{pft}_dGPP_M",
            f"{pft}_dGPP_C",
            f"{pft}_dGPP_A",
        ]

        denom = df[cols].sum(axis=1)

        df[f"{pft}_signed_GPPmax_%"] = safe_divide(
            df[f"{pft}_dGPP_M"], denom
        ) * 100.0

        df[f"{pft}_signed_CUP_%"] = safe_divide(
            df[f"{pft}_dGPP_C"], denom
        ) * 100.0

        df[f"{pft}_signed_alpha_%"] = safe_divide(
            df[f"{pft}_dGPP_A"], denom
        ) * 100.0

    return df


def summarize_signed_weighted(df, group_cols=GROUP_COLS):
    rows_eco = []
    rows_pft_gpp = []
    rows_pft_mca = []

    for keys, sub in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)

        base = dict(zip(group_cols, keys))

        # ----------------------------------------------------
        # 1) Ecosystem MCA
        # ----------------------------------------------------
        eco_cols = [
            "ecosystem_dGPP_M",
            "ecosystem_dGPP_C",
            "ecosystem_dGPP_A",
        ]

        res = weighted_signed_group(sub, eco_cols)

        row = base.copy()
        row["GPPmax_signed_weighted_%"] = res["ecosystem_dGPP_M"]
        row["CUP_signed_weighted_%"] = res["ecosystem_dGPP_C"]
        row["alpha_signed_weighted_%"] = res["ecosystem_dGPP_A"]
        row["sum_signed_weighted_%"] = (
            row["GPPmax_signed_weighted_%"]
            + row["CUP_signed_weighted_%"]
            + row["alpha_signed_weighted_%"]
        )
        rows_eco.append(row)

        # ----------------------------------------------------
        # 2) PFT GPP contribution to total PFT GPP change
        # ----------------------------------------------------
        pft_cols = [
            "tree_dGPP",
            "shrub_dGPP",
            "sphagnum_dGPP",
        ]

        res = weighted_signed_group(sub, pft_cols)

        row = base.copy()
        row["tree_signed_weighted_%"] = res["tree_dGPP"]
        row["shrub_signed_weighted_%"] = res["shrub_dGPP"]
        row["sphagnum_signed_weighted_%"] = res["sphagnum_dGPP"]
        row["sum_signed_weighted_%"] = (
            row["tree_signed_weighted_%"]
            + row["shrub_signed_weighted_%"]
            + row["sphagnum_signed_weighted_%"]
        )
        rows_pft_gpp.append(row)

        # ----------------------------------------------------
        # 3) PFT internal MCA
        # ----------------------------------------------------
        for pft in PFT_LIST:
            cols = [
                f"{pft}_dGPP_M",
                f"{pft}_dGPP_C",
                f"{pft}_dGPP_A",
            ]

            res = weighted_signed_group(sub, cols)

            row = base.copy()
            row["PFT"] = pft
            row["GPPmax_signed_weighted_%"] = res[f"{pft}_dGPP_M"]
            row["CUP_signed_weighted_%"] = res[f"{pft}_dGPP_C"]
            row["alpha_signed_weighted_%"] = res[f"{pft}_dGPP_A"]
            row["sum_signed_weighted_%"] = (
                row["GPPmax_signed_weighted_%"]
                + row["CUP_signed_weighted_%"]
                + row["alpha_signed_weighted_%"]
            )

            rows_pft_mca.append(row)

    return (
        pd.DataFrame(rows_eco),
        pd.DataFrame(rows_pft_gpp),
        pd.DataFrame(rows_pft_mca),
    )


# ============================================================
# 5. MAIN
# ============================================================

def main():

    df = pd.read_excel(IN_FILE)

    # ========================================================
    # Method 1: absolute relative contribution
    # ========================================================

    df_abs = df.copy()
    df_abs = add_absolute_relative_detail(df_abs)

    abs_eco, abs_pft_gpp, abs_pft_mca = summarize_absolute_mean_std(
        df_abs,
        group_cols=GROUP_COLS,
    )

    abs_detail_file = os.path.join(OUT_DIR, "relative_abs_detail.xlsx")
    abs_eco_file = os.path.join(OUT_DIR, "abs_ecosystem_MCA.xlsx")
    abs_pft_gpp_file = os.path.join(OUT_DIR, "abs_PFT_GPP_to_GPP.xlsx")
    abs_pft_mca_file = os.path.join(OUT_DIR, "abs_PFT_MCA.xlsx")

    df_abs.to_excel(abs_detail_file, index=False)
    abs_eco.to_excel(abs_eco_file, index=False)
    abs_pft_gpp.to_excel(abs_pft_gpp_file, index=False)
    abs_pft_mca.to_excel(abs_pft_mca_file, index=False)

    # ========================================================
    # Method 2: signed relative contribution
    # ========================================================

    df_sign = df.copy()
    df_sign = add_signed_relative_detail(df_sign)

    sign_eco, sign_pft_gpp, sign_pft_mca = summarize_signed_weighted(
        df_sign,
        group_cols=GROUP_COLS,
    )

    sign_detail_file = os.path.join(OUT_DIR, "relative_signed_detail.xlsx")
    sign_eco_file = os.path.join(OUT_DIR, "signed_ecosystem_MCA.xlsx")
    sign_pft_gpp_file = os.path.join(OUT_DIR, "signed_PFT_GPP_to_GPP.xlsx")
    sign_pft_mca_file = os.path.join(OUT_DIR, "signed_PFT_MCA.xlsx")

    df_sign.to_excel(sign_detail_file, index=False)
    sign_eco.to_excel(sign_eco_file, index=False)
    sign_pft_gpp.to_excel(sign_pft_gpp_file, index=False)
    sign_pft_mca.to_excel(sign_pft_mca_file, index=False)

    print("Saved files:")
    print(abs_detail_file)
    print(abs_eco_file)
    print(abs_pft_gpp_file)
    print(abs_pft_mca_file)
    print(sign_detail_file)
    print(sign_eco_file)
    print(sign_pft_gpp_file)
    print(sign_pft_mca_file)


if __name__ == "__main__":
    main()

Saved files:
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/relative_abs_detail.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/abs_ecosystem_MCA.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/abs_PFT_GPP_to_GPP.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/abs_PFT_MCA.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/relative_signed_detail.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/signed_ecosystem_MCA.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/signed_PFT_GPP_to_GPP.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/signed_PFT_MCA.xlsx


In [3]:
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIG
# ============================================================

IN_FILE = "../../data_results/2_results/2-5_1_contribution_25/all_results_LMDI_detail.xlsx"

OUT_DIR = "../../data_results/2_results/2-5_1_contribution_25/relative_contribution"
os.makedirs(OUT_DIR, exist_ok=True)

SPS_ECO = "ecosystem"
PFT_LIST = ["tree", "shrub", "sphagnum"]
COMP_LIST = ["M", "C", "A"]

GROUP_COLS = ["co2"]  # 可改为 ["co2", "warming"]


# ============================================================
# 2. BASIC FUNCTIONS
# ============================================================

def safe_divide(num, den):
    return np.where(den != 0, num / den, np.nan)


def weighted_signed_group(sub, comp_cols):
    """
    ξ_x = Σ_i(ΔGPP_x,i * |ΔGPP_i| / ΔGPP_i) / Σ_i|ΔGPP_i| * 100

    where:
        ΔGPP_i = sum of selected components for each row.
    """

    comps = sub[comp_cols].astype(float)
    dGPP_i = comps.sum(axis=1)

    valid = dGPP_i != 0
    denom = np.abs(dGPP_i[valid]).sum()

    out = {}

    if denom == 0:
        for col in comp_cols:
            out[col] = np.nan
        return out

    for col in comp_cols:
        numerator = (
            comps.loc[valid, col]
            * np.abs(dGPP_i[valid])
            / dGPP_i[valid]
        ).sum()

        out[col] = numerator / denom * 100.0

    return out


def weighted_signed_to_given_total(sub, comp_cols, total_col):
    """
    Calculate contribution of each component to a given total response.

    ξ_x = Σ_i(Δx_i * |ΔGPP_total_i| / ΔGPP_total_i)
          / Σ_i |ΔGPP_total_i| * 100

    This is useful for:
        PFT-component contribution to ecosystem GPP response.

    Here:
        Δx_i = PFT component contribution, e.g. tree_dGPP_M
        ΔGPP_total_i = ecosystem_dGPP or sum of all PFT components
    """

    total = sub[total_col].astype(float)

    valid = total != 0
    denom = np.abs(total[valid]).sum()

    out = {}

    if denom == 0:
        for col in comp_cols:
            out[col] = np.nan
        return out

    for col in comp_cols:
        numerator = (
            sub.loc[valid, col].astype(float)
            * np.abs(total[valid])
            / total[valid]
        ).sum()

        out[col] = numerator / denom * 100.0

    return out


# ============================================================
# 3. METHOD 1: ABSOLUTE RELATIVE CONTRIBUTION
# ============================================================

def add_absolute_relative_detail(df):
    """
    Row-level absolute relative contribution.

    RC_x = |x| / sum(|components|) * 100
    """

    # --------------------------------------------------------
    # 1) Ecosystem MCA
    # --------------------------------------------------------
    eco_cols = [
        "ecosystem_dGPP_M",
        "ecosystem_dGPP_C",
        "ecosystem_dGPP_A",
    ]

    denom = df[eco_cols].abs().sum(axis=1)

    df["ecosystem_abs_GPPmax_%"] = safe_divide(
        df["ecosystem_dGPP_M"].abs(), denom
    ) * 100.0

    df["ecosystem_abs_CUP_%"] = safe_divide(
        df["ecosystem_dGPP_C"].abs(), denom
    ) * 100.0

    df["ecosystem_abs_alpha_%"] = safe_divide(
        df["ecosystem_dGPP_A"].abs(), denom
    ) * 100.0

    # --------------------------------------------------------
    # 2) PFT GPP contribution to total PFT GPP change
    # --------------------------------------------------------
    pft_gpp_cols = [f"{pft}_dGPP" for pft in PFT_LIST]
    denom = df[pft_gpp_cols].abs().sum(axis=1)

    for pft in PFT_LIST:
        df[f"{pft}_abs_to_PFTsum_GPP_%"] = safe_divide(
            df[f"{pft}_dGPP"].abs(), denom
        ) * 100.0

    df["PFT_sum_dGPP"] = df[pft_gpp_cols].sum(axis=1)
    df["PFT_sum_minus_ecosystem_dGPP"] = (
        df["PFT_sum_dGPP"] - df["ecosystem_dGPP"]
    )

    # --------------------------------------------------------
    # 3) PFT internal MCA
    # --------------------------------------------------------
    for pft in PFT_LIST:
        cols = [
            f"{pft}_dGPP_M",
            f"{pft}_dGPP_C",
            f"{pft}_dGPP_A",
        ]

        denom = df[cols].abs().sum(axis=1)

        df[f"{pft}_abs_GPPmax_%"] = safe_divide(
            df[f"{pft}_dGPP_M"].abs(), denom
        ) * 100.0

        df[f"{pft}_abs_CUP_%"] = safe_divide(
            df[f"{pft}_dGPP_C"].abs(), denom
        ) * 100.0

        df[f"{pft}_abs_alpha_%"] = safe_divide(
            df[f"{pft}_dGPP_A"].abs(), denom
        ) * 100.0

    return df


def summarize_absolute_mean_std(df, group_cols=GROUP_COLS):
    rows_eco = []
    rows_pft_gpp = []
    rows_pft_mca = []

    for keys, sub in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)

        base = dict(zip(group_cols, keys))

        # ----------------------------------------------------
        # 1) Ecosystem MCA
        # ----------------------------------------------------
        row = base.copy()

        for var in ["GPPmax", "CUP", "alpha"]:
            col = f"ecosystem_abs_{var}_%"
            row[f"{var}_mean_%"] = sub[col].mean()
            row[f"{var}_std_%"] = sub[col].std()

        rows_eco.append(row)

        # ----------------------------------------------------
        # 2) PFT GPP contribution
        # ----------------------------------------------------
        row = base.copy()

        for pft in PFT_LIST:
            col = f"{pft}_abs_to_PFTsum_GPP_%"
            row[f"{pft}_mean_%"] = sub[col].mean()
            row[f"{pft}_std_%"] = sub[col].std()

        rows_pft_gpp.append(row)

        # ----------------------------------------------------
        # 3) PFT internal MCA
        # ----------------------------------------------------
        for pft in PFT_LIST:
            row = base.copy()
            row["PFT"] = pft

            for var in ["GPPmax", "CUP", "alpha"]:
                col = f"{pft}_abs_{var}_%"
                row[f"{var}_mean_%"] = sub[col].mean()
                row[f"{var}_std_%"] = sub[col].std()

            rows_pft_mca.append(row)

    return (
        pd.DataFrame(rows_eco),
        pd.DataFrame(rows_pft_gpp),
        pd.DataFrame(rows_pft_mca),
    )


def summarize_absolute_pft_component_to_ecosystem(df, group_cols=GROUP_COLS):
    """
    Absolute contribution of each PFT component to ecosystem GPP response.

    RC_{pft,comp}^{eco}
        = mean over rows:
          |pft_dGPP_comp| / sum_all(|pft_dGPP_comp|) * 100

    This is magnitude-based and all PFT-component terms sum to 100% per row.
    """

    rows = []

    pft_comp_cols = [
        f"{pft}_dGPP_{comp}"
        for pft in PFT_LIST
        for comp in COMP_LIST
    ]

    for keys, sub in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)

        base = dict(zip(group_cols, keys))

        denom = sub[pft_comp_cols].abs().sum(axis=1)
        row = base.copy()

        for pft in PFT_LIST:
            for comp in COMP_LIST:
                col = f"{pft}_dGPP_{comp}"
                vals = safe_divide(sub[col].abs(), denom) * 100.0
                row[f"{pft}_{comp}_abs_toEco_%_mean"] = np.nanmean(vals)
                row[f"{pft}_{comp}_abs_toEco_%_std"] = np.nanstd(vals, ddof=1)

        mean_cols = [
            f"{pft}_{comp}_abs_toEco_%_mean"
            for pft in PFT_LIST
            for comp in COMP_LIST
        ]

        row["sum_abs_toEco_%_mean"] = np.nansum([row[c] for c in mean_cols])

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# 4. METHOD 2: SIGNED RELATIVE CONTRIBUTION
# ============================================================

def add_signed_relative_detail(df):
    """
    Row-level signed relative contribution.

    RC_x_signed = x / sum(components) * 100

    It keeps sign and can be >100% or <0%.
    """

    # --------------------------------------------------------
    # 1) Ecosystem MCA
    # --------------------------------------------------------
    eco_cols = [
        "ecosystem_dGPP_M",
        "ecosystem_dGPP_C",
        "ecosystem_dGPP_A",
    ]

    denom = df[eco_cols].sum(axis=1)

    df["ecosystem_signed_GPPmax_%"] = safe_divide(
        df["ecosystem_dGPP_M"], denom
    ) * 100.0

    df["ecosystem_signed_CUP_%"] = safe_divide(
        df["ecosystem_dGPP_C"], denom
    ) * 100.0

    df["ecosystem_signed_alpha_%"] = safe_divide(
        df["ecosystem_dGPP_A"], denom
    ) * 100.0

    # --------------------------------------------------------
    # 2) PFT GPP contribution to total PFT GPP change
    # --------------------------------------------------------
    pft_gpp_cols = [f"{pft}_dGPP" for pft in PFT_LIST]
    denom = df[pft_gpp_cols].sum(axis=1)

    for pft in PFT_LIST:
        df[f"{pft}_signed_to_PFTsum_GPP_%"] = safe_divide(
            df[f"{pft}_dGPP"], denom
        ) * 100.0

    # --------------------------------------------------------
    # 3) PFT internal MCA
    # --------------------------------------------------------
    for pft in PFT_LIST:
        cols = [
            f"{pft}_dGPP_M",
            f"{pft}_dGPP_C",
            f"{pft}_dGPP_A",
        ]

        denom = df[cols].sum(axis=1)

        df[f"{pft}_signed_GPPmax_%"] = safe_divide(
            df[f"{pft}_dGPP_M"], denom
        ) * 100.0

        df[f"{pft}_signed_CUP_%"] = safe_divide(
            df[f"{pft}_dGPP_C"], denom
        ) * 100.0

        df[f"{pft}_signed_alpha_%"] = safe_divide(
            df[f"{pft}_dGPP_A"], denom
        ) * 100.0

    return df


def summarize_signed_weighted(df, group_cols=GROUP_COLS):
    """
    Existing signed summaries:
      1. ecosystem MCA
      2. PFT GPP contribution
      3. PFT internal MCA
    """

    rows_eco = []
    rows_pft_gpp = []
    rows_pft_mca = []

    for keys, sub in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)

        base = dict(zip(group_cols, keys))

        # ----------------------------------------------------
        # 1) Ecosystem MCA
        # ----------------------------------------------------
        eco_cols = [
            "ecosystem_dGPP_M",
            "ecosystem_dGPP_C",
            "ecosystem_dGPP_A",
        ]

        res = weighted_signed_group(sub, eco_cols)

        row = base.copy()
        row["GPPmax_signed_weighted_%"] = res["ecosystem_dGPP_M"]
        row["CUP_signed_weighted_%"] = res["ecosystem_dGPP_C"]
        row["alpha_signed_weighted_%"] = res["ecosystem_dGPP_A"]
        row["sum_signed_weighted_%"] = (
            row["GPPmax_signed_weighted_%"]
            + row["CUP_signed_weighted_%"]
            + row["alpha_signed_weighted_%"]
        )

        rows_eco.append(row)

        # ----------------------------------------------------
        # 2) PFT GPP contribution to total PFT GPP change
        # ----------------------------------------------------
        pft_cols = [
            "tree_dGPP",
            "shrub_dGPP",
            "sphagnum_dGPP",
        ]

        res = weighted_signed_group(sub, pft_cols)

        row = base.copy()
        row["tree_signed_weighted_%"] = res["tree_dGPP"]
        row["shrub_signed_weighted_%"] = res["shrub_dGPP"]
        row["sphagnum_signed_weighted_%"] = res["sphagnum_dGPP"]
        row["sum_signed_weighted_%"] = (
            row["tree_signed_weighted_%"]
            + row["shrub_signed_weighted_%"]
            + row["sphagnum_signed_weighted_%"]
        )

        rows_pft_gpp.append(row)

        # ----------------------------------------------------
        # 3) PFT internal MCA
        # ----------------------------------------------------
        for pft in PFT_LIST:
            cols = [
                f"{pft}_dGPP_M",
                f"{pft}_dGPP_C",
                f"{pft}_dGPP_A",
            ]

            res = weighted_signed_group(sub, cols)

            row = base.copy()
            row["PFT"] = pft
            row["GPPmax_signed_weighted_%"] = res[f"{pft}_dGPP_M"]
            row["CUP_signed_weighted_%"] = res[f"{pft}_dGPP_C"]
            row["alpha_signed_weighted_%"] = res[f"{pft}_dGPP_A"]
            row["sum_signed_weighted_%"] = (
                row["GPPmax_signed_weighted_%"]
                + row["CUP_signed_weighted_%"]
                + row["alpha_signed_weighted_%"]
            )

            rows_pft_mca.append(row)

    return (
        pd.DataFrame(rows_eco),
        pd.DataFrame(rows_pft_gpp),
        pd.DataFrame(rows_pft_mca),
    )


def summarize_signed_pft_component_to_ecosystem(df, group_cols=GROUP_COLS):
    """
    New output:
    Signed contribution of each PFT component to ecosystem GPP response.

    Formula:

        ξ_{pft,k}^{eco}
        =
        Σ_i[
            ΔGPP_{pft,k,i}
            * |ΔGPP_{eco,i}|
            / ΔGPP_{eco,i}
        ]
        /
        Σ_i |ΔGPP_{eco,i}|
        * 100

    where:
        k = M, C, A

    This directly answers:
        How much does Tree-M, Tree-C, Tree-A,
        Shrub-M, ..., Sphagnum-A contribute to ecosystem GPP response?
    """

    rows = []

    pft_comp_cols = [
        f"{pft}_dGPP_{comp}"
        for pft in PFT_LIST
        for comp in COMP_LIST
    ]

    for keys, sub in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)

        base = dict(zip(group_cols, keys))

        # Prefer ecosystem_dGPP if available.
        # Otherwise use sum of all PFT components.
        if "ecosystem_dGPP" in sub.columns:
            dGPP_eco = sub["ecosystem_dGPP"].astype(float)
        else:
            dGPP_eco = sub[pft_comp_cols].astype(float).sum(axis=1)

        valid = dGPP_eco != 0
        denom = np.abs(dGPP_eco[valid]).sum()

        row = base.copy()

        if denom == 0:
            for pft in PFT_LIST:
                for comp in COMP_LIST:
                    row[f"{pft}_{comp}_signed_toEco_%"] = np.nan
        else:
            for pft in PFT_LIST:
                for comp in COMP_LIST:
                    col = f"{pft}_dGPP_{comp}"

                    numerator = (
                        sub.loc[valid, col].astype(float)
                        * np.abs(dGPP_eco[valid])
                        / dGPP_eco[valid]
                    ).sum()

                    row[f"{pft}_{comp}_signed_toEco_%"] = numerator / denom * 100.0

        signed_cols = [
            f"{pft}_{comp}_signed_toEco_%"
            for pft in PFT_LIST
            for comp in COMP_LIST
        ]

        row["sum_signed_toEco_%"] = np.nansum([row[c] for c in signed_cols])

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# 5. MAIN
# ============================================================

def main():

    df = pd.read_excel(IN_FILE)

    # --------------------------------------------------------
    # Basic checks
    # --------------------------------------------------------
    required_cols = [
        "ecosystem_dGPP",
        "ecosystem_dGPP_M",
        "ecosystem_dGPP_C",
        "ecosystem_dGPP_A",
    ]

    required_cols += [f"{pft}_dGPP" for pft in PFT_LIST]
    required_cols += [
        f"{pft}_dGPP_{comp}"
        for pft in PFT_LIST
        for comp in COMP_LIST
    ]

    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise KeyError(f"Missing columns in input file: {missing}")

    # ========================================================
    # Method 1: absolute relative contribution
    # ========================================================

    df_abs = df.copy()
    df_abs = add_absolute_relative_detail(df_abs)

    abs_eco, abs_pft_gpp, abs_pft_mca = summarize_absolute_mean_std(
        df_abs,
        group_cols=GROUP_COLS,
    )

    abs_pft_comp_to_eco = summarize_absolute_pft_component_to_ecosystem(
        df_abs,
        group_cols=GROUP_COLS,
    )

    abs_detail_file = os.path.join(OUT_DIR, "relative_abs_detail.xlsx")
    abs_eco_file = os.path.join(OUT_DIR, "abs_ecosystem_MCA.xlsx")
    abs_pft_gpp_file = os.path.join(OUT_DIR, "abs_PFT_GPP_to_GPP.xlsx")
    abs_pft_mca_file = os.path.join(OUT_DIR, "abs_PFT_MCA.xlsx")
    abs_pft_comp_to_eco_file = os.path.join(
        OUT_DIR,
        "abs_PFT_component_to_ecosystem_GPP.xlsx"
    )

    df_abs.to_excel(abs_detail_file, index=False)
    abs_eco.to_excel(abs_eco_file, index=False)
    abs_pft_gpp.to_excel(abs_pft_gpp_file, index=False)
    abs_pft_mca.to_excel(abs_pft_mca_file, index=False)
    abs_pft_comp_to_eco.to_excel(abs_pft_comp_to_eco_file, index=False)

    # ========================================================
    # Method 2: signed relative contribution
    # ========================================================

    df_sign = df.copy()
    df_sign = add_signed_relative_detail(df_sign)

    sign_eco, sign_pft_gpp, sign_pft_mca = summarize_signed_weighted(
        df_sign,
        group_cols=GROUP_COLS,
    )

    sign_pft_comp_to_eco = summarize_signed_pft_component_to_ecosystem(
        df_sign,
        group_cols=GROUP_COLS,
    )

    sign_detail_file = os.path.join(OUT_DIR, "relative_signed_detail.xlsx")
    sign_eco_file = os.path.join(OUT_DIR, "signed_ecosystem_MCA.xlsx")
    sign_pft_gpp_file = os.path.join(OUT_DIR, "signed_PFT_GPP_to_GPP.xlsx")
    sign_pft_mca_file = os.path.join(OUT_DIR, "signed_PFT_MCA.xlsx")
    sign_pft_comp_to_eco_file = os.path.join(
        OUT_DIR,
        "signed_PFT_component_to_ecosystem_GPP.xlsx"
    )

    df_sign.to_excel(sign_detail_file, index=False)
    sign_eco.to_excel(sign_eco_file, index=False)
    sign_pft_gpp.to_excel(sign_pft_gpp_file, index=False)
    sign_pft_mca.to_excel(sign_pft_mca_file, index=False)
    sign_pft_comp_to_eco.to_excel(sign_pft_comp_to_eco_file, index=False)

    print("Saved files:")
    for f in [
        abs_detail_file,
        abs_eco_file,
        abs_pft_gpp_file,
        abs_pft_mca_file,
        abs_pft_comp_to_eco_file,
        sign_detail_file,
        sign_eco_file,
        sign_pft_gpp_file,
        sign_pft_mca_file,
        sign_pft_comp_to_eco_file,
    ]:
        print(f)


if __name__ == "__main__":
    main()

Saved files:
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/relative_abs_detail.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/abs_ecosystem_MCA.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/abs_PFT_GPP_to_GPP.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/abs_PFT_MCA.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/abs_PFT_component_to_ecosystem_GPP.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/relative_signed_detail.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/signed_ecosystem_MCA.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/signed_PFT_GPP_to_GPP.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/signed_PFT_MCA.xlsx
../../data_results/2_results/2-5_1_contribution_25/relative_contribution/signed_PFT_component_to_eco